### Collect

In [ ]:
from dotenv import load_dotenv
import requests
import json
import time
import os

load_dotenv()
auth_token = os.environ.get('AUTH_TOKEN')


url = "https://api.twitter.com/2/tweets/"


def get_tweets_by_id(id: str):
    querystring = {f"ids":{id},"tweet.fields":"author_id,created_at,geo,id,in_reply_to_user_id,lang,text","expansions":"author_id,geo.place_id,in_reply_to_user_id".format(id)}

    headers = {
        "Authorization": "Bearer " + auth_token
    }

    response = requests.request("GET", url, headers=headers, params=querystring)
   
    tweets = json.loads(response.content)

    return tweets

In [ ]:
import pandas as pd 

ids_trump_tweets = pd.read_csv('./data/ids.csv')

tweets = list()
for id in ids_trump_tweets.ids:
    response = (get_tweets_by_id(id))
    if 'data' in response:
       tweets.append(response['data'][0])

trump_tweets = pd.DataFrame(tweets)
trump_tweets

### Preprocess

In [3]:
import nltk
from nltk import tokenize
import numpy as np 
from string import punctuation
import unidecode
#stemmer = nltk.RSLPStemmer()


def proccess_text(tweets):
    
    # Removing links, mentions and hashtags
    tweets['processed_text'] = tweets.text.str.replace(r'(http\S+)', '',regex=True) \
                                          .str.replace(r'@[\w]*', '',regex=True) \
                                          .str.replace(r'#[\w]*','',regex=True) 
    print('[ok] - Removing links.')
    print('[ok] - Removing mentions.')
    print('[ok] - Removing hashtags.')

    textWords = ' '.join([text for text in tweets.processed_text])

    # Removing accent
    textWords = [unidecode.unidecode(text) for text in tweets.processed_text ]    
    print('[ok] - Removing accent.')
    
    # Creating a list of words and characters (stopwords) to be removed from the text
    # stopWords = nltk.corpus.stopwords.words("portuguese")    
    print('[ok] - Creating a list of words and characters (stopwords) to be removed from the text.')
    
    
    # Separating punctuation from words
    punctSeparator = tokenize.WordPunctTokenizer()
    punctuationList = list()
    for punct in punctuation:
        punctuationList.append(punct)
        
    #stopWords =   punctuationList + stopWords    
    stopWords =   punctuationList
    #print('[ok] - Separating punctuation from words.')


    # Iterating over the text and removing stop words 
    trasnformedText = list()    
    for text in textWords:
        newText = list()   
        text = text.lower()
        textWords = punctSeparator.tokenize(text)
        for words in textWords:
             if words not in stopWords:
                #newText.append(stemmer.stem(words))
                newText.append(words)
        trasnformedText.append(' '.join(newText))
    tweets.processed_text = trasnformedText
    print('[ok] - Removing punctuation and set text to lowecase.')
   
    # Removing all non-text characters
    tweets.processed_text = tweets['processed_text'].str.replace(r"[^a-zA-Z#]", " ", regex=True)                                                         
    print('[ok] - Removing all non-text characters.')
   
    trasnformedText = list()
    for phrase in tweets.processed_text:
        newPhrase = list()   
        newPhrase.append(' '.join(phrase.split()))
        for words in newPhrase:
            trasnformedText.append(''.join(newPhrase))
    tweets.processed_text = trasnformedText
    
    # Removing tweets with less than three terms
    index=[x for x in tweets.index if tweets.processed_text[x].count(' ') < 3]
    tweets = tweets.drop(index)
    print('[ok] - Removing tweets with less than three terms.')

    # Removing empty lines
    removeEmpty  = tweets.processed_text != ' '
    tweets = tweets[removeEmpty]
    print('[ok] - Removing empty lines.')

    tweets.reset_index(inplace=True)
    tweets = {'created_at': tweets.created_at, 'id':tweets.id,'author_id':tweets.author_id,'in_reply_to_user_id':tweets.in_reply_to_user_id, 'text': tweets.processed_text}
    #tweets = {'text': tweets.processed_text,'stance':tweets.stance}
    tweets = pd.DataFrame(tweets)
    tweets = tweets.sort_values(['created_at']).reset_index().drop(columns=["index"])
    #tweets = tweets.reset_index().drop(columns=["index"])
    
    return tweets

In [5]:
import pandas as pd 

train = pd.read_csv('./data/trump_semeval_tweets.csv')

In [6]:
tweets = proccess_text(train)

[ok] - Removing links.
[ok] - Removing mentions.
[ok] - Removing hashtags.
[ok] - Removing accent.
[ok] - Creating a list of words and characters (stopwords) to be removed from the text.
[ok] - Removing punctuation and set text to lowecase.
[ok] - Removing all non-text characters.
[ok] - Removing tweets with less than three terms.
[ok] - Removing empty lines.


In [10]:
tweets.to_csv('./data/train_test.csv')